# PINNacle Burgers 1D: Vanilla PINN + L-BFGS

Reproducción de Burgers1D-C forward del benchmark PINNacle con el baseline Vanilla: cinco capas ocultas de 100 neuronas, tanh, Glorot normal, suma uniforme de pérdidas MSE y la receta oficial Adam (lr=1e-3) con cambio a L-BFGS en la época 5,000; total 20,000 épocas.

Se ejecutan cinco seeds fijos (0–4). La métrica principal usa Cole–Hopf con Gauss–Hermite de orden 128 en una grilla común de 1025 posiciones por 129 tiempos. La referencia FEM/COMSOL oficial se puntúa aparte cuando está disponible.

Requiere Kaggle GPU T4 x2 y corre dos seeds en paralelo, una por GPU. El watchdog limita el trabajo a 14400 segundos y reserva 300 segundos para guardar JSON, CSV y el manifiesto. Cualquier corrida incompleta queda etiquetada como truncated; el estado global es complete solo si terminaron las cinco seeds con la evaluación Cole–Hopf.


## Configuración y muestreo

Se resuelve u_t + u u_x = nu u_xx en x∈[-1,1], t∈[0,1], nu=0.01/pi, con u(x,0)=-sin(pi*x) y u(-1,t)=u(1,t)=0.

El código oficial fija 8192 puntos de dominio, 2048 de frontera y 2048 iniciales. El desglose es 8192 interiores + 2048 puntos de frontera (entre ambos extremos espaciales) + 2048 puntos iniciales. El DeepXDE incluido en el repositorio concatena los tres conjuntos para el residual PDE: la MSE de la ecuación usa 12 288 coordenadas, y las MSE inicial y de frontera usan 2048 puntos cada una.

Se reproduce el sampler Hammersley predeterminado del código oficial con la secuencia Van der Corput base 2 y la coordenada temporal equiespaciada. Los tres conjuntos permanecen fijos durante cada seed. El paper informa tres repeticiones; se usan cinco por requisito del proyecto y no se presenta esta estadística como réplica exacta. L-BFGS es un brazo separado del código oficial y no forma parte de Vanilla Adam.


## Fuentes y salidas

Paper: [PINNacle, NeurIPS 2024](https://proceedings.neurips.cc/paper_files/paper/2024/file/8c63299fb2820ef41cb05e2ff11836f5-Paper-Datasets_and_Benchmarks_Track.pdf), [arXiv 2306.08827](https://arxiv.org/abs/2306.08827). Código: [i207M/PINNacle](https://github.com/i207M/PINNacle), commit 6f8d69c9c84c41d19644e4ceebdc64c4a11d3adf.

Cada seed produce un JSON con estado, épocas, pérdidas, puntuaciones, tiempos y hashes; un CSV guarda el historial por checkpoints. summary.csv incluye los cinco estados. manifest.json registra versiones, hardware, configuración, hashes de protocolo, puntos, grilla y FEM, y procedencia.

El evaluator principal no requiere Internet. El FEM opcional descarga ref/burgers1d.dat del commit fijado, valida su hash Git y reproduce el L2RE oficial sobre 101 posiciones por 11 tiempos. Si la descarga falla o el hash no coincide, se omite solo esa métrica secundaria.


In [ ]:
import csv, hashlib, io, json, math, os, platform, re, threading, time, urllib.request
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

# Required Kaggle accelerator: NvidiaTeslaT4 is T4 x2.
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError("Se requieren dos GPUs CUDA; selecciona GPU T4 x2 en Kaggle.")
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(2)]
if any("t4" not in name.lower() for name in GPU_NAMES):
    raise RuntimeError(f"Se requieren dos Tesla T4; detectadas: {GPU_NAMES}")
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

COMMIT = "6f8d69c9c84c41d19644e4ceebdc64c4a11d3adf"
REPO = "https://github.com/i207M/PINNacle"
PAPER = "https://proceedings.neurips.cc/paper_files/paper/2024/file/8c63299fb2820ef41cb05e2ff11836f5-Paper-Datasets_and_Benchmarks_Track.pdf"
SEEDS, NU = [0, 1, 2, 3, 4], 0.01 / math.pi
ND, NB, NI = 8192, 2048, 2048
WIDTH, DEPTH, LR, EPOCHS = 100, 5, 1e-3, 20000
LBFGS_SWITCH_EPOCH, LBFGS_LR, LBFGS_MAX_ITER = 5000, 1.0, 20
GH_ORDER, NX, NT = 128, 1025, 129
SAVE_EVERY, EVAL_EVERY = 1000, 2000
CAP, RESERVE = 14400, 300
FEM_URL = f"https://raw.githubusercontent.com/i207M/PINNacle/{COMMIT}/ref/burgers1d.dat"
FEM_BLOB_SHA1 = "341792e6fc1ed1f2b4de0652a53135f3e40e9326"
SOURCE_BLOBS = {
    "benchmark.py": "2066aec1c283b9eb7c0a81ade52adb1aef54e998",
    "src/pde/burgers.py": "923ce2c2ad4c0e0290ee1bbd3a8922c489d9107d",
    "src/pde/baseclass.py": "b9252bf5758133af8781b5df8d578dea12f10fa9",
    "src/optimizer/adam_lbfgs.py": "562f82fe9f3cb45dc16cc5eccd908cf70cb7f3c2",
    "deepxde/data/pde.py": "c2b65d914a5ba3e0873d25673233cb6980b99981",
    "deepxde/geometry/sampler.py": "7b384eddc0d3763cc3663eed627dfba006665759",
    "deepxde/geometry/timedomain.py": "9a78a6e5c2f1df2f590704c928db896d7f401943",
    "deepxde/nn/pytorch/nn.py": "51f56776d352e38a66179b7d60a9313b94583434",
    "src/utils/callbacks.py": "3562c4886687990f6dd685c29d9c899495fe98fc",
    "ref/burgers1d.dat": FEM_BLOB_SHA1,
}
PROTOCOL = {
    "problem": "Burgers1D-C", "nu": NU, "x": [-1, 1], "t": [0, 1],
    "ic": "-sin(pi*x)", "bc": "u(-1,t)=u(1,t)=0",
    "n_domain": ND, "n_boundary": NB, "n_initial": NI,
    "pde_residual_points": ND + NB + NI,
    "network": {"hidden_layers": DEPTH, "width": WIDTH, "activation": "tanh",
                "initializer": "Glorot normal, zero bias"},
    "loss": "unweighted sum of PDE, initial and boundary MSE",
    "optimizer": {"name": "PINNacle Adam_LBFGS", "switch_epoch": LBFGS_SWITCH_EPOCH, "adam_lr": LR, "adam_betas": [0.9, 0.999], "lbfgs_lr": LBFGS_LR, "lbfgs_max_iter": LBFGS_MAX_ITER}, "epochs": EPOCHS, "seeds": SEEDS,
    "evaluation": {"grid_t_by_x": [NT, NX], "Gauss-Hermite order": GH_ORDER},
    "official_commit": COMMIT,
}
PROTOCOL_HASH = hashlib.sha256(
    json.dumps(PROTOCOL, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

def atomic_text(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(value, encoding="utf-8")
    temp.replace(path)

def save_json(path, value):
    atomic_text(path, json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\n")

def array_hash(value, dtype):
    return hashlib.sha256(np.asarray(value, dtype=dtype, order="C").tobytes()).hexdigest()

def vdc(index):
    index = np.asarray(index, dtype=np.int64).copy()
    value, place = np.zeros(index.shape, dtype=np.float64), 2.0
    while np.any(index):
        value += (index % 2) / place
        index //= 2
        place *= 2.0
    return value

def hammersley_1d(n):
    return vdc(np.arange(1, n + 1, dtype=np.int64)).reshape(-1, 1)

def hammersley_2d(n):
    i = np.arange(1, n + 1, dtype=np.int64)
    return np.column_stack((vdc(i), i / (n + 1.0)))

def cole_hopf(x, times):
    nodes, gh_weights = np.polynomial.hermite.hermgauss(GH_ORDER)
    log_weights = np.log(gh_weights)
    exact = np.empty((len(times), len(x)), dtype=np.float64)
    exact[0] = -np.sin(np.pi * x)
    for j, t in enumerate(times[1:], start=1):
        eta = x[:, None] + 2.0 * np.sqrt(NU * t) * nodes[None, :]
        # Initial Cole-Hopf potential; common factors cancel in the ratio.
        log_phi = (1.0 - np.cos(np.pi * eta)) / (2.0 * np.pi * NU)
        terms = log_phi + log_weights[None, :]
        weights = np.exp(terms - np.max(terms, axis=1, keepdims=True))
        exact[j] = np.sum(weights * (-np.sin(np.pi * eta)), axis=1) / np.sum(weights, axis=1)
    return exact

def score(prediction, truth):
    truth = np.asarray(truth, dtype=np.float64).reshape(-1)
    error = np.asarray(prediction, dtype=np.float64).reshape(-1) - truth
    mse, mae = float(np.mean(error**2)), float(np.mean(np.abs(error)))
    return {
        "mse": mse, "mae": mae, "max_abs_error": float(np.max(np.abs(error))),
        "l1_relative_error": float(mae / np.mean(np.abs(truth))),
        "l2_relative_error": float(np.sqrt(mse / np.mean(truth**2))),
        "crmse": float(abs(np.mean(error))), "n_points": int(truth.size),
    }

def read_fem():
    try:
        with urllib.request.urlopen(FEM_URL, timeout=15) as response:
            raw = response.read()
        git_sha = hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()
        if git_sha != FEM_BLOB_SHA1:
            raise ValueError(f"FEM Git hash mismatch: {git_sha}")
        text = raw.decode("utf-8")
        data = np.loadtxt(io.BytesIO(raw), comments="%")
        header = next(line for line in text.splitlines() if line.startswith("%") and line.count("@") == data.shape[1] - 1)
        times = np.array([float(v) for v in re.findall(r"@\s*t\s*=\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)", header)])
        if len(times) + 1 != data.shape[1]:
            raise ValueError("FEM header and columns disagree")
        xx, tt = np.meshgrid(data[:, 0], times, indexing="ij")
        return {
            "meta": {"available": True, "url": FEM_URL, "git_blob_sha1": git_sha,
                     "sha256": hashlib.sha256(raw).hexdigest(), "n_x": data.shape[0],
                     "n_t": len(times), "n_points": int(data[:, 1:].size), "times": times.tolist()},
            "points": np.column_stack((xx.ravel(), tt.ravel())).astype(np.float32),
            "truth": data[:, 1:].reshape(-1, 1).astype(np.float64),
        }
    except Exception as exc:
        return {"meta": {"available": False, "url": FEM_URL,
                         "expected_git_blob_sha1": FEM_BLOB_SHA1,
                         "reason": f"{type(exc).__name__}: {exc}"},
                "points": None, "truth": None}

def make_model(seed):
    generator = torch.Generator(device="cpu").manual_seed(int(seed))
    sizes = [2] + [WIDTH] * DEPTH + [1]
    modules = []
    for i, (nin, nout) in enumerate(zip(sizes[:-1], sizes[1:])):
        layer = torch.nn.Linear(nin, nout, dtype=torch.float32)
        torch.nn.init.xavier_normal_(layer.weight, generator=generator)
        torch.nn.init.zeros_(layer.bias)
        modules.append(layer)
        if i < len(sizes) - 2:
            modules.append(torch.nn.Tanh())
    return torch.nn.Sequential(*modules)

def model_hash(model):
    h = hashlib.sha256()
    for name, value in model.state_dict().items():
        h.update(name.encode())
        h.update(value.detach().cpu().numpy().astype("<f4", copy=False).tobytes())
    return h.hexdigest()

def train_points(seed):
    rng = np.random.RandomState(seed)
    uv = hammersley_2d(ND)
    domain = np.column_stack((-1.0 + 2.0 * uv[:, 0], uv[:, 1])).astype(np.float32)
    ix = (-1.0 + 2.0 * hammersley_1d(NI)[:, 0]).astype(np.float32)
    initial = np.column_stack((ix, np.zeros(NI))).astype(np.float32)
    bx = rng.choice(np.array([-1.0, 1.0]), size=NB)
    bt = rng.permutation(hammersley_1d(NB)[:, 0])
    boundary = np.column_stack((bx, bt)).astype(np.float32)
    residual = np.concatenate((initial, domain, boundary), axis=0)
    hashes = {
        "domain_sha256": array_hash(domain, "<f4"),
        "initial_sha256": array_hash(initial, "<f4"),
        "boundary_sha256": array_hash(boundary, "<f4"),
        "pde_coordinates_sha256": array_hash(residual, "<f4"),
    }
    return residual, initial, boundary, hashes

def predict(model, points, device):
    model.eval()
    pieces = []
    with torch.no_grad():
        for start in range(0, len(points), 32768):
            batch = torch.as_tensor(points[start:start+32768], dtype=torch.float32, device=device)
            pieces.append(model(batch).cpu().numpy())
    torch.cuda.synchronize(device)
    return np.concatenate(pieces).astype(np.float64)

def evaluate(model, device, points, exact, fem):
    primary = score(predict(model, points, device), exact.reshape(-1, 1))
    secondary = score(predict(model, fem["points"], device), fem["truth"]) if fem["meta"]["available"] else None
    return primary, secondary

def save_seed_csv(path, rows):
    columns = ["epoch", "status", "loss_pde", "loss_initial", "loss_boundary",
               "loss_total", "l2re_cole_hopf", "l2re_fem", "epoch_seconds", "elapsed_seconds"]
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
    atomic_text(path, buf.getvalue())

# The 7200-second clock includes reference construction, FEM retrieval, training and output.
started = time.monotonic()
deadline, train_deadline = started + CAP, started + CAP - RESERVE
run_utc = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
root = Path("/kaggle/working")
if not root.exists():
    root = Path.cwd()
run_dir = root / "pinnacle-reference-burgers-lbfgs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)

x_eval, t_eval = np.linspace(-1, 1, NX), np.linspace(0, 1, NT)
exact = cole_hopf(x_eval, t_eval)
gx, gt = np.meshgrid(x_eval, t_eval, indexing="xy")
eval_points = np.column_stack((gx.ravel(), gt.ravel())).astype(np.float32)
fem = read_fem()

manifest = {
    "schema_version": "1.0", "run_id": run_id, "run_started_utc": run_utc,
    "run_status": "running", "wall_clock_cap_seconds": CAP,
    "finalization_reserve_seconds": RESERVE, "protocol": PROTOCOL,
    "protocol_sha256": PROTOCOL_HASH,
    "source": {"paper_url": PAPER, "repository_url": REPO, "commit": COMMIT,
               "file_git_blob_sha1": SOURCE_BLOBS},
    "fem_reference": fem["meta"],
    "evaluation_grid": {"n_x": NX, "n_t": NT, "shape_t_by_x": [NT, NX],
                        "quadrature": "Gauss-Hermite", "order": GH_ORDER,
                        "cole_hopf_sha256_f64": array_hash(exact, "<f8"),
                        "x_sha256_f64": array_hash(x_eval, "<f8"),
                        "t_sha256_f64": array_hash(t_eval, "<f8")},
    "environment": {"python": platform.python_version(), "platform": platform.platform(),
                    "numpy": np.__version__, "torch": torch.__version__,
                    "cuda_runtime": torch.version.cuda, "gpu_names": GPU_NAMES,
                    "gpu_count": torch.cuda.device_count(),
                    "kaggle_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown")},
    "seed_runs": {str(s): {"status": "pending", "epochs_completed": 0} for s in SEEDS},
}
lock, stop = threading.Lock(), threading.Event()

def persist(seed, record, rows):
    save_json(run_dir / f"seed_{seed}.json", record)
    save_seed_csv(run_dir / f"seed_{seed}.csv", rows)
    with lock:
        manifest["global_elapsed_seconds"] = float(time.monotonic() - started)
        manifest["seed_runs"][str(seed)] = {
            "status": record["status"], "epochs_completed": record["epochs_completed"],
            "elapsed_seconds": record.get("elapsed_seconds", 0.0),
            "cole_hopf_l2re": record["metrics_cole_hopf"]["l2_relative_error"] if record.get("metrics_cole_hopf") else None,
            "fem_l2re": record["metrics_fem"]["l2_relative_error"] if record.get("metrics_fem") else None,
        }
        save_json(run_dir / "manifest.json", manifest)

save_json(run_dir / "manifest.json", manifest)

def train_seed(seed, device_name):
    device, t0, model = torch.device(device_name), time.monotonic(), None
    record = {
        "schema_version": "1.0", "seed": int(seed), "status": "running",
        "device": device_name, "requested_epochs": EPOCHS, "epochs_completed": 0,
        "optimizer": "PINNacle Adam_LBFGS", "switch_epoch": LBFGS_SWITCH_EPOCH, "learning_rate": LR,
        "loss_pde_final": None, "loss_initial_final": None, "loss_boundary_final": None,
        "loss_total_final": None, "metrics_cole_hopf": None, "metrics_fem": None,
        "elapsed_seconds": 0.0, "epochs_per_second": None,
        "protocol_sha256": PROTOCOL_HASH, "sampling_hashes": None,
        "initial_model_sha256": None, "final_model_sha256": None, "error": None,
    }
    rows = []
    persist(seed, record, rows)
    try:
        if time.monotonic() >= train_deadline:
            record.update(status="truncated", error="No time budget remained before start.")
            persist(seed, record, rows)
            return record
        r_np, i_np, b_np, hashes = train_points(seed)
        record["sampling_hashes"] = hashes
        residual = torch.as_tensor(r_np, dtype=torch.float32, device=device)
        initial = torch.as_tensor(i_np, dtype=torch.float32, device=device)
        boundary = torch.as_tensor(b_np, dtype=torch.float32, device=device)
        model = make_model(seed).to(device)
        record["initial_model_sha256"] = model_hash(model)
        adam_optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.999))
        lbfgs_optimizer = torch.optim.LBFGS(model.parameters(), lr=LBFGS_LR, max_iter=LBFGS_MAX_ITER)
        persist(seed, record, rows)
        last_loss, last_saved = [None] * 4, 0

        def loss_closure(active_optimizer):
            active_optimizer.zero_grad(set_to_none=True)
            pde_x = residual.detach().clone().requires_grad_(True)
            u = model(pde_x)
            grad = torch.autograd.grad(u, pde_x, torch.ones_like(u), create_graph=True)[0]
            ux, ut = grad[:, 0:1], grad[:, 1:2]
            uxx = torch.autograd.grad(ux, pde_x, torch.ones_like(ux), create_graph=True)[0][:, 0:1]
            lpde = torch.mean((ut + u * ux - NU * uxx).square())
            lic = torch.mean((model(initial) + torch.sin(math.pi * initial[:, 0:1])).square())
            lbc = torch.mean(model(boundary).square())
            loss = lpde + lic + lbc
            total = float(loss.detach().item())
            if not math.isfinite(total):
                raise FloatingPointError("Non-finite PINNacle loss.")
            loss.backward()
            last_loss[:] = [float(v.detach().item()) for v in (lpde, lic, lbc, loss)]
            return loss

        for epoch in range(1, EPOCHS + 1):
            if stop.is_set() or time.monotonic() >= train_deadline:
                stop.set()
                record["status"] = "truncated"
                break
            epoch_t0 = time.monotonic()
            if epoch < LBFGS_SWITCH_EPOCH:
                loss_closure(adam_optimizer)
                adam_optimizer.step()
            else:
                lbfgs_optimizer.step(lambda: loss_closure(lbfgs_optimizer))
                if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
                    loss_closure(lbfgs_optimizer)
            total = last_loss[3]
            if not math.isfinite(total):
                record.update(status="failed", error=f"Non-finite loss at epoch {epoch}.")
                break
            torch.cuda.synchronize(device)
            record["epochs_completed"] = epoch

            if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
                if epoch % EVAL_EVERY == 0 and time.monotonic() + 20 < train_deadline:
                    record["metrics_cole_hopf"], record["metrics_fem"] = evaluate(
                        model, device, eval_points, exact, fem
                    )
                elapsed = float(time.monotonic() - t0)
                rows.append({
                    "epoch": epoch, "status": "running", "loss_pde": last_loss[0],
                    "loss_initial": last_loss[1], "loss_boundary": last_loss[2],
                    "loss_total": last_loss[3],
                    "l2re_cole_hopf": record["metrics_cole_hopf"]["l2_relative_error"] if record["metrics_cole_hopf"] else None,
                    "l2re_fem": record["metrics_fem"]["l2_relative_error"] if record["metrics_fem"] else None,
                    "epoch_seconds": float(time.monotonic() - epoch_t0),
                    "elapsed_seconds": elapsed,
                })
                record.update(loss_pde_final=last_loss[0], loss_initial_final=last_loss[1],
                              loss_boundary_final=last_loss[2], loss_total_final=last_loss[3],
                              elapsed_seconds=elapsed, epochs_per_second=epoch / max(elapsed, 1e-12))
                last_saved = epoch
                persist(seed, record, rows)

        if record["status"] == "running":
            record["status"] = "complete" if record["epochs_completed"] == EPOCHS else "truncated"
        if record["epochs_completed"] > last_saved:
            rows.append({
                "epoch": record["epochs_completed"], "status": record["status"],
                "loss_pde": last_loss[0], "loss_initial": last_loss[1],
                "loss_boundary": last_loss[2], "loss_total": last_loss[3],
                "l2re_cole_hopf": record["metrics_cole_hopf"]["l2_relative_error"] if record["metrics_cole_hopf"] else None,
                "l2re_fem": record["metrics_fem"]["l2_relative_error"] if record["metrics_fem"] else None,
                "epoch_seconds": None, "elapsed_seconds": float(time.monotonic() - t0),
            })
        if record["status"] == "complete" and record["metrics_cole_hopf"] is None:
            if time.monotonic() + 20 < train_deadline:
                record["metrics_cole_hopf"], record["metrics_fem"] = evaluate(
                    model, device, eval_points, exact, fem
                )
            else:
                record.update(status="truncated", error="Final Cole-Hopf evaluation missed watchdog reserve.")
        record["elapsed_seconds"] = float(time.monotonic() - t0)
        record["epochs_per_second"] = record["epochs_completed"] / max(record["elapsed_seconds"], 1e-12)
        record["final_model_sha256"] = model_hash(model)
        if rows:
            rows[-1]["status"] = record["status"]
            rows[-1]["elapsed_seconds"] = record["elapsed_seconds"]
            if record["metrics_cole_hopf"]:
                rows[-1]["l2re_cole_hopf"] = record["metrics_cole_hopf"]["l2_relative_error"]
            if record["metrics_fem"]:
                rows[-1]["l2re_fem"] = record["metrics_fem"]["l2_relative_error"]
        persist(seed, record, rows)
        return record
    except Exception as exc:
        record.update(status="failed", error=f"{type(exc).__name__}: {exc}",
                      elapsed_seconds=float(time.monotonic() - t0))
        if model is not None:
            record["final_model_sha256"] = model_hash(model)
        persist(seed, record, rows)
        return record

# Keep one job per free GPU and stop scheduling at the global deadline.
active, records, seed_iter = {}, {}, iter(SEEDS)
pool = ThreadPoolExecutor(max_workers=2)
try:
    for gpu in range(2):
        seed = next(seed_iter, None)
        if seed is not None:
            future = pool.submit(train_seed, seed, f"cuda:{gpu}")
            active[future] = (seed, gpu)
    while active:
        done, _ = wait(active, timeout=1.0, return_when=FIRST_COMPLETED)
        if not done:
            if time.monotonic() >= train_deadline:
                stop.set()
            continue
        for future in done:
            seed, gpu = active.pop(future)
            records[str(seed)] = future.result()
            next_seed = next(seed_iter, None)
            if next_seed is not None and not stop.is_set() and time.monotonic() < train_deadline:
                next_future = pool.submit(train_seed, next_seed, f"cuda:{gpu}")
                active[next_future] = (next_seed, gpu)
except KeyboardInterrupt:
    stop.set()
finally:
    pool.shutdown(wait=True, cancel_futures=True)

# Never omit seeds which did not start before the deadline.
for seed in SEEDS:
    if manifest["seed_runs"][str(seed)]["status"] == "pending":
        record = {
            "schema_version": "1.0", "seed": seed, "status": "truncated",
            "device": None, "requested_epochs": EPOCHS, "epochs_completed": 0,
            "elapsed_seconds": 0.0, "metrics_cole_hopf": None, "metrics_fem": None,
            "protocol_sha256": PROTOCOL_HASH,
            "error": "Not started before the watchdog stopped scheduling seeds.",
        }
        persist(seed, record, [])
statuses = [manifest["seed_runs"][str(s)]["status"] for s in SEEDS]
manifest["run_status"] = "complete" if all(s == "complete" for s in statuses) else (
    "truncated" if any(s in ("pending", "running", "truncated") for s in statuses) else "failed"
)
manifest["global_elapsed_seconds"] = float(time.monotonic() - started)
manifest["run_finished_utc"] = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
save_json(run_dir / "manifest.json", manifest)

fields = ["seed", "status", "epochs_completed", "elapsed_seconds", "epochs_per_second",
          "loss_pde_final", "loss_initial_final", "loss_boundary_final", "loss_total_final",
          "cole_hopf_l2re", "fem_l2re", "device", "final_model_sha256"]
buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=fields)
writer.writeheader()
for seed in SEEDS:
    rec = json.loads((run_dir / f"seed_{seed}.json").read_text(encoding="utf-8"))
    writer.writerow({
        "seed": seed, "status": rec["status"], "epochs_completed": rec["epochs_completed"],
        "elapsed_seconds": rec.get("elapsed_seconds"), "epochs_per_second": rec.get("epochs_per_second"),
        "loss_pde_final": rec.get("loss_pde_final"), "loss_initial_final": rec.get("loss_initial_final"),
        "loss_boundary_final": rec.get("loss_boundary_final"), "loss_total_final": rec.get("loss_total_final"),
        "cole_hopf_l2re": rec["metrics_cole_hopf"]["l2_relative_error"] if rec.get("metrics_cole_hopf") else None,
        "fem_l2re": rec["metrics_fem"]["l2_relative_error"] if rec.get("metrics_fem") else None,
        "device": rec.get("device"), "final_model_sha256": rec.get("final_model_sha256"),
    })
atomic_text(run_dir / "summary.csv", buf.getvalue())
print(f"Estado global: {manifest['run_status']}")
print(f"Tiempo total: {manifest['global_elapsed_seconds']:.1f} s / {CAP} s")
print(f"Resultados: {run_dir}")
for seed in SEEDS:
    rec = json.loads((run_dir / f"seed_{seed}.json").read_text(encoding="utf-8"))
    val = rec["metrics_cole_hopf"]["l2_relative_error"] if rec.get("metrics_cole_hopf") else None
    print(f"seed={seed} status={rec['status']} epochs={rec['epochs_completed']} Cole-Hopf L2RE={val}")
